# GoogleNet(含并行连接的网络)

GoogLeNet吸收了NiN中串联网络的思想，并在此基础上做了改进。使用不同大小的卷积核组合是有利的，这里介绍一下一个稍微简化的GoogleNet网络

## Inception块

Inception有四条并行的路径组成，前三条路径使用窗口大小为$1 \times 1, 3 \times 3, 5 \times 5$的卷积层，从不同的空间大小中提取信息，中间的两条路径在输入上执行$1 \times 1$的卷积层以减少通道数，从而降低模型的复杂性。 第四条路径使用$3 \times 3$的卷积层，提取信息， 最后用$1 \times 1$卷积块去修改通道数， 这四条路径都使用合适的填充来使输入与输出的高和宽一致，最后我们将每条线路的输出在通道维度上连结，并构成Inception块的输出

Inception块中调整的超参数是每层的输出通道数

网络中的$1 \times 1$卷积层大幅减少了参数量，同时保留了通道信息，为后续的特征提取提供了基础。

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l


class Inception(nn.Module):
    # c1--c4是每条路径的输出通道数
    def __init__(self, in_channels, c1, c2, c3, c4, **kwargs):  # **kwargs 用于传递可变参数，不用写元组的符号了，c1，c2,c3,c4是每个路径的通道数
        super(Inception, self).__init__(**kwargs)  # 调用Inception类的父类的__init__方法，把关键字参数传递给他
        # 线路1，单1x1卷积层
        self.p1_1 = nn.Conv2d(in_channels, c1, kernel_size=1)
        # 线路2，1x1卷积层后接3x3卷积层
        self.p2_1 = nn.Conv2d(in_channels, c2[0], kernel_size=1)
        self.p2_2 = nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1)
        # 线路3，1x1卷积层后接5x5卷积层
        self.p3_1 = nn.Conv2d(in_channels, c3[0], kernel_size=1)
        self.p3_2 = nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2)
        # 线路4，3x3最大汇聚层后接1x1卷积层
        self.p4_1 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.p4_2 = nn.Conv2d(in_channels, c4, kernel_size=1)

    def forward(self, x):  # 总之每一层之后都要加一个Relu
        p1 = F.relu(self.p1_1(x))  # 路径1的输出
        p2 = F.relu(self.p2_2(F.relu(self.p2_1(x))))  # 路径2的输出
        p3 = F.relu(self.p3_2(F.relu(self.p3_1(x))))
        p4 = F.relu(self.p4_2(self.p4_1(x)))
        # 在通道维度上连结输出
        return torch.cat((p1, p2, p3, p4), dim=1)  # 四条路在通道维度数为1（输出通道数维度）拼接起来

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Inception块得到的Concatenation跟input是等同高宽的，高宽不会变，变的只是通道数（每条路上的通道数可以是不相同的）

原始论文里面给的是192个通道，Concatenation给出的是256个通道

googleNet分出了5个stage组合（高宽减半是一个stage）和VGG块有一些相似

比较AlexNet，在接入Inception之前，需要迅速的降低高宽，扩展通道数

## 实现每一个stage

In [2]:
b1 = nn.Sequential(nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
                   nn.ReLU(),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
b2 = nn.Sequential(nn.Conv2d(64, 64, kernel_size=1),
                   nn.ReLU(),
                   nn.Conv2d(64, 192, kernel_size=3, padding=1),
                   nn.ReLU(),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
b3 = nn.Sequential(Inception(192, 64, (96, 128), (16, 32), 32),
                   Inception(256, 128, (128, 192), (32, 96), 64),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
b4 = nn.Sequential(Inception(480, 192, (96, 208), (16, 48), 64),
                   Inception(512, 160, (112, 224), (24, 64), 64),
                   Inception(512, 128, (128, 256), (24, 64), 64),
                   Inception(512, 112, (144, 288), (32, 64), 64),
                   Inception(528, 256, (160, 320), (32, 128), 128),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
b5 = nn.Sequential(Inception(832, 256, (160, 320), (32, 128), 128),
                   Inception(832, 384, (192, 384), (48, 128), 128),
                   nn.AdaptiveAvgPool2d((1,1)),
                   nn.Flatten())

net = nn.Sequential(b1, b2, b3, b4, b5, nn.Linear(1024, 10)) # 最后接全连接层去输出

把输入的高宽降到96，看一下各层网络的输出形状

In [3]:
X = torch.rand(size=(1, 1, 96, 96))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape:\t', X.shape)

Sequential output shape:	 torch.Size([1, 64, 24, 24])
Sequential output shape:	 torch.Size([1, 192, 12, 12])
Sequential output shape:	 torch.Size([1, 480, 6, 6])
Sequential output shape:	 torch.Size([1, 832, 3, 3])
Sequential output shape:	 torch.Size([1, 1024])
Linear output shape:	 torch.Size([1, 10])
